# Data structures and cohomology in `fastop`

This tutorial shows how to bring finite simplicial data into `fastop`, compute cohomology over $\mathbf F_p$, multiply classes, and apply Steenrod operations. The three input classes share one cohomology interface but encode different kinds of geometry.


In [ ]:
from pathlib import Path
import sys

repo = Path.cwd()
if not (repo / 'src' / 'fastop').exists():
    for candidate in (repo.parent, repo.parent.parent):
        if (candidate / 'src' / 'fastop').exists():
            repo = candidate
            break
sys.path.insert(0, str(repo / 'src'))

from fastop import DeltaComplex, SimplexReference, SimplicialComplex, SimplicialSet, spaces


## 1. Choose the representation that matches the data

- Use `SimplicialComplex` for an ordinary triangulation with globally named vertices.
- Use `DeltaComplex` when cells are indexed and faces may repeat, as in polygon identifications and quotients.
- Use `SimplicialSet` when degeneracies, Cartesian products, or symmetric powers matter.

Most users should begin with the first class unless their source data already consists of face maps.


### Abstract simplicial complexes

Pass the maximal simplices as tuples of integer vertex labels. `fastop` generates every face automatically.


In [ ]:
circle = SimplicialComplex([(0, 1), (1, 2), (0, 2)])
print('dimension:', circle.dimension)
print('vertices:', circle.vertices)
print('edges:', circle.cells(1))
assert circle.cohomology(p=3).betti_numbers() == {0: 1, 1: 1}


### Delta-complexes from dense face maps

A degree-$d$ cell has exactly $d+1$ face indices. The following circle has one vertex and one loop edge; both edge faces are vertex `0`.


In [ ]:
delta_circle = DeltaComplex([
    [()],
    [(0, 0)],
])
assert delta_circle.f_vector() == (1, 1)
assert delta_circle.cohomology(p=3).betti_numbers() == {0: 1, 1: 1}


### Simplicial sets and degeneracies

`SimplexReference(degree, cell, operator)` records a possibly degenerate face. Repeated values in `operator` encode degeneracies. Direct construction is advanced; adapters and built-in models are usually simpler. This minimal model of $\mathbf{RP}^2$ has one cell in every degree $0,1,2$.


In [ ]:
degenerate_edge = SimplexReference(0, 0, (0, 0))
projective_plane = SimplicialSet.from_face_maps([
    [()],
    [(0, 0)],
    [(0, degenerate_edge, 0)],
])
assert projective_plane.f_vector() == (1, 1, 1)
assert projective_plane == spaces.nonorientable_surface(1)


## 2. The common cohomology interface

Every model supplies `cohomology(p=...)`. The returned object computes degree data lazily: asking for $H^d$ constructs only the adjacent boundary maps needed in degree $d$.


In [ ]:
CP3 = spaces.minimal_simplicial_sphere(2).symmetric_power(3)
H = CP3.cohomology(p=3)
print('Betti numbers:', H.betti_numbers())
u = H.basis(2)[0]
print('class:', u)
representative = u.cocycle()
print('chosen cocycle support:', len(representative))
print('first terms:', list(representative.items())[:5])


A basis is deterministic for a fixed cell ordering, but names such as `h^2,0` are coordinates rather than canonical geometric labels. Use equality and ring operations instead of assuming that a one-dimensional target basis has a preferred sign.


## 3. Cup products and powers

Cohomology classes multiply with `*`; nonnegative powers use `**`. Internally this is the Alexander–Whitney product followed by projection into the chosen cohomology basis.

$$
(f\smile g)(\sigma)=
f(\sigma|_{[0,\ldots,r]})\,g(\sigma|_{[r,\ldots,r+s]}).
$$


In [ ]:
assert H.one() * u == u
assert u**2
assert u**3 == H.basis(6)[0] or u**3 == 2 * H.basis(6)[0]
print('u^2 =', u**2)
print('u^3 =', u**3)
print('H^2 x H^2 product columns:', H.cup_product_matrix(2, 2))


## 4. Steenrod operations

At $p=2$, `x.operation(k)` and `x.sq(k)` compute $Sq^k(x)$. At an odd prime, `x.operation(r)` computes $P^r(x)$ and `bockstein=True` selects $\beta P^r$. The top-power identity is now directly testable in the ring.


In [ ]:
assert u.operation(1) == u**3
assert H.operation_rank(2, 1) == 1

H2 = projective_plane.cohomology(p=2)
a = H2.basis(1)[0]
assert a.sq(1) == a**2 == H2.basis(2)[0]


### About the Bockstein

`operation(0, bockstein=True)` already computes the mod-$p$ Bockstein $\beta=\beta P^0$. A separate `bockstein()` method would therefore only be an alias. A broader API for Bocksteins associated to other coefficient sequences can wait until `fastop` supports coefficients beyond prime fields.
